# Observabilidad: la caja negra que sí puedes mirar

**Facsímil 6 · Construir y operar** — capítulo 4 (observabilidad: logs, métricas, trazas y costes).

Un sistema de IA sin observabilidad es una caja negra que solo te avisa de sus problemas... por boca de
los usuarios enfadados. En este cuaderno pones un servicio «en producción» (simulado), registras cada
petición con su latencia, sus tokens y su coste, y construyes el panel que de verdad te dice si el
sistema va bien. Por el camino descubres por qué **mirar la media te engaña**, por qué el percentil 95
es el número que te quita el sueño, cómo se reparte el gasto entre tus usuarios (mal: unos pocos se lo
llevan casi todo), cómo se ve un **incidente en directo** y cómo decidir cuándo despertar a alguien.
Medir latencia, coste y errores por petición es lo que separa *operar* de *rezar*.

### Qué vas a aprender
- Los tres pilares de la observabilidad: **logs** (qué pasó), **métricas** (cuánto) y **trazas** (por
  dónde).
- A registrar peticiones y resumirlas en métricas que llevan a decisiones.
- Por qué la **media miente** y los **percentiles** (p95, p99) cuentan la verdad de quien peor lo pasa.
- A leer la **forma** de la distribución de latencias y a verla dibujada.
- A definir **objetivos de servicio** (SLO), **alertas** y un **presupuesto de error** mensual.
- A repartir el coste **por usuario** y **por tipo de petición**, y a descubrir quién te cuesta dinero.
- A reconocer un **incidente** en una serie temporal y a desglosar una petición con una **traza**.
- Los **cuatro golden signals** que resumen la salud de cualquier servicio.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves: todo está simulado.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(7)

# Cuatro tipos de peticion ("endpoints"), cada uno con su latencia base y su rango de tokens.
ENDPOINTS = {
    "chat":       {"lat": 500,  "tok": (300, 900)},
    "resumen":    {"lat": 1200, "tok": (800, 2500)},
    "traduccion": {"lat": 350,  "tok": (150, 600)},
    "busqueda":   {"lat": 250,  "tok": (80, 300)},
}
N = 4000
nombres_ep = list(ENDPOINTS)
endpoint = np.random.choice(nombres_ep, N, p=[0.45, 0.20, 0.20, 0.15])

# Latencia: una base por endpoint... y una COLA LENTA (8%) que sufre picos (sobrecarga, reintentos).
latencia_ms = np.array([np.random.normal(ENDPOINTS[e]["lat"], ENDPOINTS[e]["lat"]*0.25) for e in endpoint])
cola = np.random.random(N) < 0.08
latencia_ms[cola] += np.random.normal(2000, 600, cola.sum())
latencia_ms = np.clip(latencia_ms, 40, None)

tokens = np.array([np.random.randint(*ENDPOINTS[e]["tok"]) for e in endpoint])
coste  = tokens * (0.7/1_000_000)          # precio por token
error  = np.random.random(N) < 0.03        # 3% de errores
usuario = np.minimum(np.random.zipf(1.8, N), 300)   # pocos usuarios concentran muchas peticiones

print(f"{N} peticiones registradas. Las 3 primeras trazas:")
for i in range(3):
    print(f"  req#{i}: {endpoint[i]:<10} | {latencia_ms[i]:>6.0f} ms | {tokens[i]:>4} tok "
          f"| ${coste[i]:.6f} | usuario#{usuario[i]:<3} | {'ERROR' if error[i] else 'ok'}")


## 1. Los tres pilares: logs, métricas, trazas

Observar no es una sola cosa. Se apoya en tres tipos de datos complementarios:

- **Logs:** el registro de *qué* pasó en cada petición (entrada, salida, error). Útil para investigar un
  caso concreto, como leer el parte de un accidente.
- **Métricas:** números agregados a lo largo del tiempo (*cuánta* latencia, *cuántos* errores, *cuánto*
  coste). Útiles para ver tendencias y disparar alertas, como el cuadro de mandos del coche.
- **Trazas:** el recorrido de una petición *por dónde* pasó (qué pasos, cuánto tardó cada uno). Útiles
  para ver dónde está el cuello de botella, como seguir un paquete por sus escaneos.

Empecemos por los logs: un buen log cuenta una petición entera de un vistazo.


In [ ]:
print("LOG DEL SERVICIO (tres lineas de ejemplo)")
for i in [0, 100, 250]:
    estado = "ERROR" if error[i] else "ok"
    print(f'  {{"req": {i}, "endpoint": "{endpoint[i]}", "usuario": {usuario[i]}, '
          f'"latencia_ms": {latencia_ms[i]:.0f}, "tokens": {tokens[i]}, '
          f'"coste_usd": {coste[i]:.6f}, "estado": "{estado}"}}')
print("\nUn log por peticion sirve para INVESTIGAR un caso. Para ver el conjunto, hacen falta metricas.")


## 2. El panel: las métricas que importan

De miles de logs sueltos no se opera. Se opera de un panel con pocas métricas bien elegidas. Estas son
las que deciden si despiertas a alguien a las 3 de la mañana.


In [ ]:
print("PANEL DEL SERVICIO")
print(f"  Latencia media:   {latencia_ms.mean():>7.0f} ms")
print(f"  Latencia p50:     {np.percentile(latencia_ms,50):>7.0f} ms  (la mitad va por debajo)")
print(f"  Latencia p95:     {np.percentile(latencia_ms,95):>7.0f} ms  (el 5% peor sufre esto)")
print(f"  Latencia p99:     {np.percentile(latencia_ms,99):>7.0f} ms  (el 1% peor)")
print(f"  Coste total:      ${coste.sum():>8.4f}  ({N} peticiones)")
print(f"  Coste por 1.000:  ${1000*coste.mean():>8.4f}")
print(f"  Tasa de error:    {100*error.mean():>7.1f}%")


## 3. El número que engaña: media frente a percentiles

Mira la diferencia entre la latencia **media** y el **p95**. La media te dice que «todo va razonable»,
mientras un 5% de tus usuarios espera varias veces más. ¿Por qué pasa? Porque la media la arrastran los
muchos casos rápidos y esconde la **cola** de casos lentos. Si solo vigilas la media, no verás el
problema hasta que esos usuarios se vayan. **Operar es vigilar las colas, no los promedios.** Lo
medimos: qué fracción de usuarios sufre por encima de un umbral que la media no delata.


In [ ]:
UMBRAL_LENTO = 1500    # ms; por encima, el usuario lo percibe como "lento"
frac_lentos = 100 * (latencia_ms > UMBRAL_LENTO).mean()
print(f"Latencia MEDIA: {latencia_ms.mean():.0f} ms  -> 'parece que todo va bien'")
print(f"Pero el {frac_lentos:.0f}% de las peticiones supera los {UMBRAL_LENTO} ms y se siente lento.")
print(f"El p95 ({np.percentile(latencia_ms,95):.0f} ms) si lo delata; la media te lo esconde.")


## 4. La forma manda: la distribución de latencias, dibujada

Un solo número nunca cuenta toda la historia. La *forma* de la distribución de latencias sí: se ve el
grueso de peticiones rápidas a la izquierda y la **cola larga** de lentas estirándose a la derecha. Sobre
el dibujo marcamos la media y el p95 para que veas, de un vistazo, lo lejos que cae uno del otro.


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.hist(np.clip(latencia_ms, 0, 4000), bins=60, color="0.6")
m, p95 = latencia_ms.mean(), np.percentile(latencia_ms, 95)
ax.axvline(m,   color="black",     lw=2, label=f"media  {m:.0f} ms")
ax.axvline(p95, color="firebrick", lw=2, label=f"p95    {p95:.0f} ms")
ax.set_xlabel("latencia (ms)"); ax.set_ylabel("nº de peticiones")
ax.set_title("Distribucion de latencias: el grueso rapido y la cola lenta")
ax.legend(); plt.tight_layout(); plt.show()
print("La media cae entre el grueso; el p95 vive en la cola. Por eso vigilamos percentiles.")


## 5. Una alerta de verdad: cuándo despertar a alguien

Un panel no sirve si nadie lo mira a las 3 de la mañana. Definimos **objetivos de servicio** (SLO):
umbrales que, si se cruzan, disparan una **alerta**. Por ejemplo: «el p95 no debe pasar de 2 segundos» y
«los errores no deben superar el 5%». Comprobamos si se cumplen.


In [ ]:
SLO = {"p95_ms": 2000, "error_pct": 5.0}
medido = {"p95_ms": np.percentile(latencia_ms, 95), "error_pct": 100*error.mean()}
print("Estado frente a los SLO:")
print(f"  p95:     {medido['p95_ms']:>6.0f} ms  (objetivo <= {SLO['p95_ms']})  "
      f"-> {'[!] ALERTA' if medido['p95_ms']>SLO['p95_ms'] else 'ok'}")
print(f"  errores: {medido['error_pct']:>6.1f}%  (objetivo <= {SLO['error_pct']:.0f}%)  "
      f"-> {'[!] ALERTA' if medido['error_pct']>SLO['error_pct'] else 'ok'}")


## 6. Experimento: cómo la cola mueve el p95 (y no la media)

Para convencerte de que la media es mala consejera, agrandamos la **cola lenta** (más peticiones
tardísimas) y vemos qué le pasa a la media frente al p95. La media sube de forma moderada; el p95 se
dispara mucho más, porque captura justo a quienes peor lo pasan. Esa es la prueba de por qué se vigilan
percentiles y no solo el promedio.


In [ ]:
print("% de peticiones lentas | latencia media | p95")
for frac_cola in [0.05, 0.10, 0.20, 0.30]:
    lat = np.concatenate([np.random.normal(400, 80, int(N*(1-frac_cola))),
                          np.random.normal(2200, 600, int(N*frac_cola))])
    lat = np.clip(lat, 50, None)
    print(f"        {frac_cola*100:>2.0f}%           |   {lat.mean():>6.0f} ms    | {np.percentile(lat,95):>6.0f} ms")
print("\nLa media sube algo, pero el p95 se dispara mucho mas. Por eso un buen panel vigila la cola.")


## 7. El gasto tampoco se reparte por igual: la cola del coste

Igual que la latencia tiene cola, el **coste** la tiene: un puñado de usuarios suele concentrar buena
parte de la factura. Si no lo miras, no sabes a quién aplicar un límite, a quién cobrar un plan distinto
o quién está abusando. Agrupamos el gasto por usuario y vemos cuánto se llevan los que más consumen.


In [ ]:
gasto = np.zeros(usuario.max() + 1)
np.add.at(gasto, usuario, coste)             # suma el coste de cada usuario
orden = np.argsort(gasto)[::-1]
n_top = max(1, int(0.05 * (gasto > 0).sum()))
frac_top = 100 * gasto[orden[:n_top]].sum() / gasto.sum()
print(f"Usuarios con gasto: {(gasto>0).sum()}")
print(f"El 5% que mas gasta ({n_top} usuarios) se lleva el {frac_top:.0f}% del coste total.\n")
print("Top 3 usuarios por gasto:")
for u in orden[:3]:
    print(f"  usuario#{u:<3}: ${gasto[u]:.5f}  ({(usuario==u).sum()} peticiones)")


## 8. Coste y latencia por tipo de petición

No todas las peticiones cuestan ni tardan lo mismo. Un resumen largo gasta muchos más tokens que una
búsqueda corta. Desglosar por **endpoint** te dice dónde optimizar: a veces un solo tipo de petición se
come la mitad del gasto y ni lo sospechabas.


In [ ]:
print(f"{'endpoint':<12}{'peticiones':>11}{'lat. media':>12}{'p95':>9}{'coste total':>14}")
for e in nombres_ep:
    m = endpoint == e
    print(f"{e:<12}{m.sum():>11}{latencia_ms[m].mean():>9.0f} ms{np.percentile(latencia_ms[m],95):>6.0f} ms"
          f"   ${coste[m].sum():>10.5f}")
caro = max(nombres_ep, key=lambda e: coste[endpoint==e].sum())
print(f"\nEl endpoint que mas gasta es '{caro}': ahi rinde mas cualquier optimizacion.")


## 9. Un incidente en directo: un despliegue que salió mal

Hasta aquí, fotos fijas. La realidad es una película: las métricas cambian con el tiempo. Simulamos un
**incidente**: a partir de cierto momento (un despliegue defectuoso), la tasa de error se dispara.
Vigilamos el error en una **ventana reciente** —no en todo el histórico, que lo diluye— y comprobamos
que la alerta salta a tiempo.


In [ ]:
err_serie = np.zeros(N, dtype=bool)
corte = int(N*0.8)
err_serie[:corte]  = np.random.random(corte)   < 0.02      # antes del despliegue: 2%
err_serie[corte:]  = np.random.random(N-corte) < 0.18      # despues: 18% (incidente)
VENTANA = 400
err_global  = 100*err_serie.mean()
err_reciente = 100*err_serie[-VENTANA:].mean()
print(f"Error en TODO el historico:        {err_global:>5.1f}%  -> 'parece tolerable'")
print(f"Error en las ultimas {VENTANA} peticiones: {err_reciente:>5.1f}%  -> {'[!] ALERTA' if err_reciente>5 else 'ok'}")
print("\nLa media de todo el dia diluye el incidente. La ventana reciente lo caza enseguida.")


## 10. La degradación lenta: vigilar la tendencia

No todos los problemas llegan de golpe. A veces la latencia **sube poco a poco** (una tabla que crece,
una caché que se llena) hasta que un día cruza el umbral. Una **media móvil** suaviza el ruido y deja ver
la tendencia. Si la pendiente sube, te adelantas al problema en vez de esperar a que estalle.


In [ ]:
drift = np.linspace(0, 1, N)                  # degradacion progresiva
lat_t = np.clip(np.random.normal(400, 90, N) + 900*drift**2, 40, None)
k = 200
movil = np.convolve(lat_t, np.ones(k)/k, mode="valid")
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(lat_t, color="0.8", lw=0.6, label="latencia por peticion")
ax.plot(np.arange(k-1, N), movil, color="black", lw=2, label=f"media movil ({k})")
ax.axhline(1000, color="firebrick", ls="--", label="umbral SLO")
ax.set_xlabel("peticion (orden temporal)"); ax.set_ylabel("latencia (ms)")
ax.set_title("Degradacion lenta: la media movil revela la tendencia")
ax.legend(); plt.tight_layout(); plt.show()
print(f"Latencia media al principio: {lat_t[:500].mean():.0f} ms | al final: {lat_t[-500:].mean():.0f} ms.")


## 11. El presupuesto de error: cuánto puedes fallar este mes

Un SLO no exige la perfección: exige, por ejemplo, «99% de peticiones correctas al mes». Ese 1% restante
es tu **presupuesto de error**: un saldo que puedes gastar. Mientras quede saldo, puedes arriesgar
(desplegar, experimentar); si se agota, toca frenar y estabilizar. Lo calculamos con los errores que
hemos registrado.


In [ ]:
SLO_DISPONIBILIDAD = 0.99
presupuesto = (1 - SLO_DISPONIBILIDAD) * N      # nº de fallos permitidos
fallos = int(error.sum())
consumido = 100 * fallos / presupuesto
print(f"SLO de disponibilidad: {SLO_DISPONIBILIDAD*100:.0f}%  ->  presupuesto = {presupuesto:.0f} fallos")
print(f"Fallos registrados:    {fallos}")
print(f"Presupuesto consumido: {consumido:.0f}%  -> {'[!] AGOTADO: toca frenar y estabilizar' if consumido>100 else 'ok: queda saldo para arriesgar'}")


## 12. Trazas: ¿dónde se va el tiempo dentro de una petición?

Cuando una petición tarda, la pregunta no es «¿cuánto?», sino «¿en qué paso?». Una **traza** descompone
la petición en sus etapas y mide cada una. Casi siempre hay un **cuello de botella** que se lleva el
grueso del tiempo: ahí, y solo ahí, conviene optimizar.


In [ ]:
traza = {
    "recepcion":      12,
    "cola de espera": 85,
    "tokenizar":      18,
    "inferencia LLM": 640,
    "post-proceso":   30,
    "enviar":          9,
}
total = sum(traza.values())
cuello = max(traza, key=traza.get)
print(f"Traza de una peticion (total {total} ms):")
for paso, ms in traza.items():
    barra = "#" * int(40*ms/total)
    print(f"  {paso:<16}{ms:>4} ms  {barra}")
print(f"\nCuello de botella: '{cuello}' ({100*traza[cuello]/total:.0f}% del tiempo). Optimiza ahi primero.")


## 13. Los cuatro golden signals

Si tuvieras que vigilar solo cuatro números en cualquier servicio, serían estos (el método clásico de las
operaciones de fiabilidad): **latencia** (cuánto tarda), **tráfico** (cuánta demanda), **errores** (qué
fracción falla) y **saturación** (cómo de lleno está). Con estos cuatro detectas casi cualquier problema
sin ahogarte en métricas. Los reunimos en un único cuadro.


In [ ]:
saturacion = np.clip(np.random.normal(55, 15, N) + cola*25, 0, 100)   # % de uso simulado
print("LOS CUATRO GOLDEN SIGNALS")
print(f"  Latencia    -> p95:        {np.percentile(latencia_ms,95):>6.0f} ms")
print(f"  Trafico     -> peticiones: {N:>6}")
print(f"  Errores     -> tasa:       {100*error.mean():>6.1f}%")
print(f"  Saturacion  -> uso p95:    {np.percentile(saturacion,95):>6.0f}%")
print("\nCuatro numeros, una foto honesta de la salud del servicio.")


## 14. Pruébalo tú

1. **Endurece el SLO de latencia:** baja `p95_ms` a 1000 y vuelve a comprobar la alerta. ¿Tu servicio lo
   cumpliría con estos datos?
2. **Mueve la ventana del incidente:** cambia `VENTANA` a 100 y a 1000. Una ventana corta reacciona antes
   pero da más falsas alarmas; una larga, al revés. ¿Dónde está tu equilibrio?
3. **Cambia el reparto de endpoints:** sube la proporción de `resumen` (el que más tokens gasta) y mira
   cómo se dispara el coste total y el p95.
4. **Otro cuello de botella:** edita la `traza` para que el cuello sea «cola de espera» (servidor
   saturado) en vez de la inferencia. La cura es distinta: escalar máquinas, no cambiar de modelo.
5. **Tu propio presupuesto de error:** define un SLO del 99,9% y mira cuánto presupuesto consume el mismo
   3% de errores. Un nueve más es diez veces más exigente.
6. **Coste por usuario y por hora:** combina `gasto` con una hora simulada y busca la franja más cara del
   día. En real, ahí está el pico de capacidad que pagas.


## 15. Errores comunes

- **Mirar solo la media.** Esconde a los usuarios que peor lo pasan. Vigila percentiles (p95, p99).
- **Calcular el error sobre todo el histórico.** Diluye los incidentes. Usa ventanas recientes para
  alertar.
- **No tener alertas.** Un panel que nadie mira no sirve. Define umbrales (SLO) que avisen solos.
- **Confundir un número con la realidad.** La *forma* de la distribución y la *tendencia* en el tiempo
  cuentan cosas que ningún promedio capta.
- **Optimizar a ciegas.** Sin una traza no sabes dónde está el cuello de botella; tocas lo que no toca.
- **Olvidar el coste.** En un sistema con LLM, el coste por petición es una métrica de primera, igual que
  la latencia y los errores; y se reparte muy desigual entre usuarios y endpoints.
- **Medir todo y no decidir nada.** Las métricas existen para actuar (escalar, optimizar, revertir,
  poner límites), no para decorar.


## 16. Qué te llevas

- Observar es **registrar cada petición** (latencia, tokens, coste, error, usuario, endpoint) y resumirla
  en métricas que llevan a decisiones. Tres pilares: logs, métricas, trazas.
- La **media miente**; los **percentiles** (p95, p99), la **forma** de la distribución y la **tendencia**
  en el tiempo cuentan la verdad de quien peor lo pasa.
- Las **alertas**, los **SLO** y el **presupuesto de error** convierten un panel en acción: avisan, y
  además te dicen cuándo puedes arriesgar y cuándo toca frenar.
- El **coste** tiene cola, igual que la latencia: unos pocos usuarios y algún endpoint concentran el
  gasto. Y los **cuatro golden signals** (latencia, tráfico, errores, saturación) te dan la foto honesta
  con poquísimos números.

**Para seguir:** el siguiente cuaderno usa estas métricas para decidir a qué modelo mandar cada petición
(routing) y cuándo tirar de un plan B (fallback) sin pasarse de presupuesto; el capítulo 7 trata los
despliegues progresivos (canary, rollback) que se apoyan en este panel.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*